<a href="https://colab.research.google.com/github/darcymarsh06/ISYS2001-darcy-marsh/blob/main/Assessment_2_trialDarcy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Problem Definition

My financial assistant will run to calculate how the user is tracking to achiveing their yearly savings goal based on their yearly income, monthly spending and yearly savings goal. It will then provide a personalised output based on the conclusion of their financal results in the form of both statistics on their spending and an AI analysis and finiancial advice.

Step 2: Analysis

For this code I need to install google genai, and gradio then also include the API Key, pandas for my code to run smoothly. The code must then be able to request each peice of information from the user and give the correct output if info is invalid. The it must store this data to give the correct and personalised data in the output. The gradio interface must also be formatted correctly to look presentable and run smoothly.

Step 3: Design

psudocode throughout the code

In [2]:
# The Google Gen AI library is how our Python code talks to Gemini.
!pip install google-genai

In [7]:
from google.colab import userdata
from google import genai

# The client reads your key from Secrets, so the key itself never appears here.
client = genai.Client(api_key=userdata.get("API_KEY"))

# The flash model is fast and free-tier friendly. The exact name changes every
# few months, so check AI Studio for the current one and edit this single line.
MODEL = "gemini-3.8-flash"

print("Connected to the Gemini API.")

Connected to the Gemini API.


In [1]:
#install gradio
!pip install --upgrade gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.7.1
    Uninstalling gradio_client-2.7.1:
      Successfully uninstalled gradio_client-2.7.1
  Attempting uninstall: gradio
    Found existing installation: gradio 6.29.0
    Uninstalling gradio-6.29.0:
      Successfully uninstalled gradio-6.29.0


In [2]:
import os
import gradio as gr
import pandas as pd
from google import genai
from google.colab import userdata

# Apply the Gemini using the Colab secret 'API_KEY'
api_key = userdata.get('API_KEY')
client = genai.Client(api_key=api_key)

# CSV setup to check if user has previous entries (Load previous data or fallback template)
csv_filename = "my_expenses.csv"

if os.path.exists(csv_filename):
    initial_df = pd.read_csv(csv_filename)
    initial_text = "\n".join(
        [
            f"{row['Item']}: {row['Price']}"
            for _, row in initial_df.iterrows()
        ]
    )
else:
    initial_text = (
        "Rent: 1200\nGroceries: 450\nUtilities: 150\nEntertainment: 200"
    )


def track_finance(name, income, savings_goal, expenses_text):
    # Name and welcome
    if not name.strip():
        return "Please enter your name.", "", ""

    # Income range ($1000 to $1000000)
    if income < 1000 or income > 1000000:
        return (
            "Your yearly income must be between $1000 and $1000000",
            "",
            "",
        )

    # Sort expenses text into a list and then a Pandas DataFrame
    expense_list = []
    lines = expenses_text.strip().split("\n")
    for line in lines:
        if ":" in line:
            parts = line.split(":")
            item_name = parts[0].strip()
            try:
                item_price = float(parts[1].strip().replace("$", ""))
                if item_price > 0:
                    expense_list.append((item_name, item_price))
            except ValueError:
                continue

    # Calculate total monthly expenses manually by looping through items
        total_monthly_expense = sum(price for _, price in current_list)

        # Automatically save updated table to CSV using standard Python file writing
        try:
            with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.DictWriter(file, fieldnames=["Item", "Price"])
                writer.writeheader()
                for item, price in current_list:
                    writer.writerow({"Item": item, "Price": price})
        except Exception:
            pass
        else:
          current_list = []
          total_monthly_expense = 0.0


    # Calculations
    yearly_expense = total_monthly_expense * 12
    remaining_after_expenses = income - yearly_expense
    projected_savings_gap = remaining_after_expenses - savings_goal

    # Format the ranked expenses
    ranked_str = "### Expenses Ranked (Highest to Lowest):\n"
    if current_list:
        for idx, (item, price) in enumerate(current_list, 1):
            ranked_str += f"{idx}. {item}: ${price:,.2f} per month (${price * 12:,.2f} yearly)\n"
    else:
        ranked_str += "No valid expenses entered.\n"

    # Statistics and Goal tracking
    stats_str = "### Financial Statistics & Goal Tracking\n"
    stats_str += f"Yearly Income: ${income:,.2f}\n"
    stats_str += f"Total Monthly Expenses: ${total_monthly_expense:,.2f}\n"
    stats_str += f"Total Yearly Expenses: ${yearly_expense:,.2f}\n"
    stats_str += f"Projected Remaining: ${remaining_after_expenses:,.2f}\n"
    stats_str += f"Your Savings Goal: ${savings_goal:,.2f}\n"

    #Tracker result findings
    if remaining_after_expenses >= savings_goal:
        stats_str += f"\n Congratulations {name}! You are on track to meet your savings goal with a surplus of ${remaining_after_expenses - savings_goal:,.2f}!\n"
    else:
        stats_str += f"\n Warning {name}! You are falling short of your savings goal by ${abs(projected_savings_gap):,.2f}.\n"

    # Generate unique AI suggestions using Google Gen AI API
    prompt = f"""
    You are a friendly and personable financial assistant.
    Analyze these financial stats for {name}:
    - Yearly Income: ${income:,.2f}
    - Total Yearly Expenses: ${yearly_expense:,.2f}
    - Remaining After Expenses: ${remaining_after_expenses:,.2f}
    - Savings Goal: ${savings_goal:,.2f}
    - Top Expense: {current_list[0][0] if current_list else 'None'} (${current_list[0][1] if current_list else 0}/mo)

    Provide 2 or 3 short, personalized, and actionable financial suggestions for {name}. Keep it warm and encouraging.
    """

    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash", contents=prompt
        )
        suggestions_str = f"### Personalised AI Suggestions:\n{response.text}"
    except Exception as e:
        suggestions_str = f"### Personalised Suggestions:\n Could not generate AI suggestions at the moment. (Error: {e})"

    return ranked_str, stats_str, suggestions_str



# Build Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as finance_tracker:
    gr.Markdown("# Personal Financial Assistant & Goal Tracker")
    gr.Markdown(
        "Welcome! Enter your information below to calculate your income, track your expenses, and see how you are tracking to reach your savings goal!"
    )

    with gr.Row():
        with gr.Column():
            name_input = gr.Textbox(label="Your Name", placeholder="e.g., Darcy")
            income_input = gr.Number(
                label="Yearly Income ($)",
                value=50000,
                minimum=1000,
                maximum=1000000,
            )
            savings_goal_input = gr.Number(
                label="Yearly Savings Goal ($)", value=10000
            )
            expenses_input = gr.Textbox(
                label="Monthly Expenses (Format: Item: Price per line)",
                value=initial_text,
                placeholder=(
                    "Rent: 1200\nGroceries: 450\nUtilities:"
                    " 150\nEntertainment: 200"
                ),
                lines=5,
            )
            submit_btn = gr.Button("Calculate My Finances", variant="primary")

        with gr.Column():
            ranked_output = gr.Markdown(label="Ranked Expenses")
            stats_output = gr.Markdown(label="Statistics")
            suggestions_output = gr.Markdown(label="Suggestions")

    submit_btn.click(
        fn=track_finance,
        inputs=[
            name_input,
            income_input,
            savings_goal_input,
            expenses_input,
        ],
        outputs=[ranked_output, stats_output, suggestions_output],
    )

finance_tracker.launch(quiet=True)

/tmp/ipykernel_4349/407467620.py:125: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as finance_tracker:


* Running on public URL: https://9bc02345ef8ef1b64d.gradio.live


Step 5: Test and De-bug
I ran my code in stages developing with Google Gemini the phases of code then making corrections and additions to create my final project. Each phase of the debugging and testing is outlined in the diary. My main bugs were indentation errors and not defining data.

Step 6: Maintenance and Documentation
To maintain the code the Gradio Interface is upgraded to stay up to date and run smoothly and the google genai is run on a 3.8 flash.